# Speed Layer — Bronze Exploration (Streaming)
This notebook is an exploratory Databricks demo for the Speed (streaming) layer.
Use it to connect to Kinesis/S3 sources, inspect incoming events, and prototype writes to a Delta bronze table.

In [ ]:
# Configure interactive widgets for demo (Databricks widgets)
try:
    dbutils.widgets.text('kinesis_stream', '')
    dbutils.widgets.text('kinesis_region', 'us-west-2')
    dbutils.widgets.text('s3_path', '')
    kinesis_stream = dbutils.widgets.get('kinesis_stream')
    kinesis_region = dbutils.widgets.get('kinesis_region')
    s3_path = dbutils.widgets.get('s3_path')
    print('Widgets loaded: kinesis_stream=', kinesis_stream, 's3_path=', s3_path)
except NameError:
    print('Widgets not available outside Databricks; set variables manually when running locally.')

In [ ]:
# Inspect an S3 prefix (Autoloader demo)
# Replace s3://<BUCKET>/data/streaming with your path when running on Databricks
try:
    path = s3_path or 's3://<BUCKET>/data/streaming'
    print('Listing path:', path)
    files = dbutils.fs.ls(path)
    display(files)
except NameError:
    print('dbutils not available. Run on Databricks to inspect S3 mounts or DBFS.')

In [ ]:
# Read streaming input via Autoloader (cloudFiles) — demo pattern
# Autoloader requires Databricks Runtime with cloudFiles support
try:
    cloud_path = path
    stream = spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('header', 'true')
        .option('inferSchema', 'true')
        .load(cloud_path)
    display(stream.limit(10))
except Exception as e:
    print('Autoloader read failed — run on Databricks with correct path and permissions.
', e)

In [ ]:
# Read from Kinesis (Databricks Kinesis connector) — demo pattern
try:
    ks = kinesis_stream or '<KINESIS_STREAM>'
    kr = kinesis_region or 'us-west-2'
    kdf = spark.readStream.format('kinesis')
        .option('streamName', ks)
        .option('region', kr)
        .option('startingposition', 'TRIM_HORIZON')
        .load()
    display(kdf.limit(10))
except Exception as e:
    print('Kinesis read failed — run on Databricks with the connector installed and the stream configured.
', e)

In [ ]:
# Demo: write incoming stream to Delta bronze table using foreachBatch
try:
    def write_bronze(batch_df, batch_id):
        # Example enrichments: add ingestion metadata
        from pyspark.sql import functions as F
        batch_df = batch_df.withColumn('_ingestion_time', F.current_timestamp())
        batch_df.write.format('delta').mode('append').saveAsTable('events_bronze')

    # Choose the available stream (kdf or stream) for the demo
    input_stream = locals().get('kdf', locals().get('stream'))
    if input_stream is None:
        print('No streaming DataFrame found. Run a previous cell to create `kdf` or `stream`.')
    else:
        q = input_stream.writeStream.foreachBatch(write_bronze)
            .option('checkpointLocation', '/mnt/dbfs/checkpoints/events_bronze')
            .start()
        print('Started demo bronze write; stop the stream in Databricks when finished.')
except Exception as e:
    print('Failed to start writeStream (run on Databricks).
', e)

---
Next steps: use this notebook to validate the event schema, run small sample uploads to S3 to trigger Lambda/Kinesis, and then open `02_silver_transformation.ipynb` to prototype cleaning and partitioning.